# Distorted Visual Sequence Pattern Recognition (CRNN + CTC)

**Goal.** Read the 6-character code hidden inside each noisy, distorted grayscale image and predict it as text. Scored by **Character Error Rate (CER)** — lower is better.

**Why a CRNN + CTC?** The text is a *left-to-right sequence*, but we never get per-character boxes. A **CRNN** (Convolutional Recurrent Neural Network) solves exactly this:

1. **CNN** scans the image and produces a horizontal strip of feature vectors (one per image "column").
2. **BiLSTM** reads that strip in both directions, modelling how characters follow one another.
3. **CTC loss** lets us train with only the *final string* as the label — no need to mark *where* each character sits.

This is the standard, well-understood recipe for OCR / captcha-style recognition, and it matches the approaches the problem statement hints at (CNN + LSTM/BiLSTM/CRNN + CTC).

**Workflow:** Setup → EDA → Label encoding → Dataset & augmentation → Model → Decoding & metric → Training → Evaluation → How to improve.

## 1. Setup & configuration

All knobs live in one `CFG` object so it becomes easy to experiment and tune. The data-root finder checks common Kaggle/Colab/local locations automatically.

In [ ]:
import os, random, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k): return x


def find_data_root():
    candidates = [
        Path("data/cig_ps"), Path("../data/cig_ps"),
        Path("/kaggle/input"), Path("/content/cig_ps"),
    ]
    for c in candidates:
        if (c / "train-labels.csv").exists():
            return c
        if c.exists():
            for sub in c.rglob("train-labels.csv"):
                return sub.parent
    raise FileNotFoundError("Could not locate cig_ps. Set CFG.data_root manually.")


class CFG:
    data_root   = find_data_root()
    img_height  = 32          # CRNN needs a fixed height; width carries the sequence
    img_width   = 160
    chars       = "23456789ABCDEFGHJKMNPQRSTUVWXYZ"   # discovered in EDA below
    rnn_hidden  = 256
    dropout     = 0.1
    batch_size  = 128
    epochs      = 30
    lr          = 5e-4
    weight_decay = 1e-5
    val_fraction = 0.1
    num_workers = 0
    seed        = 42
    augment     = True
    device      = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir     = Path("outputs"); out_dir.mkdir(exist_ok=True)

CFG.train_dir  = CFG.data_root / "train_images"
CFG.test_dir   = CFG.data_root / "test_images"
CFG.labels_csv = CFG.data_root / "train-labels.csv"
CFG.num_classes = len(CFG.chars) + 1     # +1 for the CTC blank token


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(CFG.seed)
print("Device:", CFG.device, "| Data root:", CFG.data_root)

## 2. Exploratory Data Analysis (EDA)

Before modelling we look at *what we are dealing with*: how many samples, how long the labels are, which characters appear, and what the distortions look like.

In [ ]:
raw = pd.read_csv(CFG.labels_csv)[["image", "text"]].copy()
raw["text"] = raw["text"].astype(str)

# Two rows out of 20,000 were mangled by spreadsheet auto-formatting
# ("5.40E+12", "04-Mar-54"). Keep only well-formed codes from our alphabet.
valid = raw["text"].str.fullmatch(r"[23456789A-HJ-NP-Z]+")
df = raw[valid].reset_index(drop=True)

print(f"Total rows: {len(raw)} | clean rows: {len(df)} | dropped: {(~valid).sum()}")
print("Label length distribution:", df['text'].str.len().value_counts().to_dict())
charset = sorted(set("".join(df["text"])))
print(f"Charset ({len(charset)}):", "".join(charset))
print("Note: 0/1/I/L/O are intentionally absent (ambiguous with O/2/...).")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for ax, (_, row) in zip(axes.ravel(), df.sample(8, random_state=1).iterrows()):
    ax.imshow(Image.open(CFG.train_dir / row["image"]).convert("L"), cmap="gray")
    ax.set_title(row["text"]); ax.axis("off")
fig.suptitle("Sample training images (label shown above each)")
plt.tight_layout(); plt.show()

**What we learned:**

- **20,000 images, 200×100, grayscale.** Almost all labels are exactly **6 characters**.
- Distortions seen in the samples: speckle/background noise, a large black **occlusion blob**, blur, and characters that overlap or sit on irregular baselines.
- **31-symbol alphabet** (`2-9`, `A-Z` minus `I/L/O`). We reserve index `0` for the CTC **blank**, so characters map to `1..31`.

These observations justify: (a) converting to grayscale + resizing to a fixed `32×160`, (b) mild geometric **augmentation** to imitate the irregular alignment, and (c) a sequence model with CTC so we never need to localise characters.

## 3. Label encoding for CTC

CTC needs a special **blank** symbol (index `0`). A predicted string is built by taking the network's per-column guesses, **merging consecutive duplicates**, then **deleting blanks**. The blank is what lets the model output a genuine double letter (e.g. `22`) — it separates them with a blank so they are not merged.

In [ ]:
class CharsetCodec:
    def __init__(self, chars):
        self.chars = chars
        self.blank = 0
        self.c2i = {c: i + 1 for i, c in enumerate(chars)}
        self.i2c = {i + 1: c for i, c in enumerate(chars)}

    def encode(self, text):
        return [self.c2i[c] for c in text]

    def decode(self, indices):
        out, prev = [], None
        for idx in indices:
            if idx != prev and idx != self.blank:
                out.append(self.i2c.get(idx, ""))
            prev = idx
        return "".join(out)

codec = CharsetCodec(CFG.chars)
print("BU522X ->", codec.encode("BU522X"))

## 4. Dataset & augmentation

Each image is converted to grayscale, optionally given a small random rotation/shift/scale (only during training), resized to `32×160`, and normalised to roughly `[-1, 1]`. The custom `collate` function packs variable-length labels into the flat format `CTCLoss` expects.

In [ ]:
def build_transforms(augment):
    ops = [transforms.Grayscale(1)]
    if augment:
        ops.append(transforms.RandomAffine(degrees=4, translate=(0.03, 0.06), scale=(0.92, 1.0)))
    ops += [
        transforms.Resize((CFG.img_height, CFG.img_width)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ]
    return transforms.Compose(ops)


class TextImageDataset(Dataset):
    def __init__(self, image_dir, records=None, filenames=None, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.items = records if records is not None else [(f, "") for f in filenames]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        fname, label = self.items[i]
        img = Image.open(self.image_dir / fname)
        if self.transform:
            img = self.transform(img)
        target = torch.tensor(codec.encode(label), dtype=torch.long)
        return img, target, torch.tensor(len(target)), fname


def ctc_collate(batch):
    imgs, targets, lengths, names = zip(*batch)
    imgs = torch.stack(imgs, 0)
    targets = torch.cat(targets) if len(targets) else torch.tensor([], dtype=torch.long)
    lengths = torch.stack(lengths)
    return imgs, targets, lengths, list(names)

## 5. The CRNN model

- **CNN** (7 conv layers with pooling) shrinks the `32`-pixel height down to **1** while keeping width, turning the image into a sequence of `~41` feature vectors of size `512`.
- **2-layer BiLSTM** reads that sequence both ways and learns character-to-character dependencies.
- A **linear layer** scores all `32` classes (`31` characters + blank) at every step; `log_softmax` gives the log-probabilities `CTCLoss` wants.

In [ ]:
class CRNN(nn.Module):
    def __init__(self, num_classes, rnn_hidden=256, dropout=0.1):
        super().__init__()

        # BatchNorm after every conv + Kaiming init are essential: without them
        # gradients don't reach the early layers and CTC collapses to all-blanks.
        def block(in_c, out_c):
            return [nn.Conv2d(in_c, out_c, 3, 1, 1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)]

        self.cnn = nn.Sequential(
            *block(1, 64),   nn.MaxPool2d(2, 2),                  # 32x160 -> 16x80
            *block(64, 128), nn.MaxPool2d(2, 2),                 # -> 8x40
            *block(128, 256), *block(256, 256),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)),                # -> 4x41
            *block(256, 512), *block(512, 512),
            nn.MaxPool2d((2, 2), (2, 1), (0, 1)),                # -> 2x42
            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(inplace=True),  # -> 1x41
            nn.Dropout(dropout),
        )
        self.rnn = nn.LSTM(512, rnn_hidden, num_layers=2, bidirectional=True, dropout=dropout)
        self.fc = nn.Linear(rnn_hidden * 2, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        feat = self.cnn(x)                       # (B, C, 1, W)
        feat = feat.squeeze(2).permute(2, 0, 1)  # (W, B, C): W is the time axis
        seq, _ = self.rnn(feat)
        return self.fc(seq).log_softmax(2)       # (W, B, num_classes)

model = CRNN(CFG.num_classes, CFG.rnn_hidden, CFG.dropout).to(CFG.device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print("Output shape (T, B, C):", tuple(model(torch.randn(2, 1, CFG.img_height, CFG.img_width).to(CFG.device)).shape))

## 6. Decoding & the CER metric

**Greedy decoding** takes the most likely class at each of the ~41 steps, then collapses duplicates and removes blanks to get the final string.

**Character Error Rate (CER)** = (insertions + deletions + substitutions) needed to turn the prediction into the truth, divided by the truth length. It is the edit (Levenshtein) distance, normalised. `0.0` is perfect; lower is better.

In [ ]:
def greedy_decode(log_probs):
    best = log_probs.argmax(2).permute(1, 0)   # (B, W)
    return [codec.decode(seq.tolist()) for seq in best]


def _edit(a, b):
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
    return dp[n]


def char_error_rate(preds, targets):
    dist = sum(_edit(p, t) for p, t in zip(preds, targets))
    length = sum(max(len(t), 1) for t in targets)
    return dist / max(length, 1)

## 7. Training

We split off 10% for validation, then train with **Adam** and **CTCLoss**. Gradients are clipped (CTC can spike early) and the learning rate is halved when validation CER plateaus. We **save the checkpoint with the best validation CER**, so a late over-fitting epoch never hurts the final submission.

In [ ]:
data = df.sample(frac=1.0, random_state=CFG.seed).reset_index(drop=True)
n_val = int(len(data) * CFG.val_fraction)
val_df, train_df = data.iloc[:n_val], data.iloc[n_val:]

train_ds = TextImageDataset(CFG.train_dir, records=list(zip(train_df["image"], train_df["text"])),
                            transform=build_transforms(CFG.augment))
val_ds   = TextImageDataset(CFG.train_dir, records=list(zip(val_df["image"], val_df["text"])),
                            transform=build_transforms(False))

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, collate_fn=ctc_collate, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, collate_fn=ctc_collate, pin_memory=True)
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, gts = [], []
    for imgs, targets, lengths, _ in loader:
        preds += greedy_decode(model(imgs.to(CFG.device)).cpu())
        off = 0
        for n in lengths.tolist():
            gts.append("".join(codec.i2c[i] for i in targets[off:off + n].tolist())); off += n
    return char_error_rate(preds, gts), preds, gts

In [ ]:
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

history = {"train_loss": [], "val_cer": []}
best_cer = float("inf")
ckpt_path = CFG.out_dir / "crnn_best.pt"

for epoch in range(1, CFG.epochs + 1):
    model.train()
    running = 0.0
    for imgs, targets, target_lengths, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{CFG.epochs}", leave=False):
        imgs, targets, target_lengths = imgs.to(CFG.device), targets.to(CFG.device), target_lengths.to(CFG.device)
        log_probs = model(imgs)
        T, B, _ = log_probs.size()
        input_lengths = torch.full((B,), T, dtype=torch.long, device=CFG.device)
        loss = criterion(log_probs, targets, input_lengths, target_lengths)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running += loss.item() * B

    train_loss = running / len(train_ds)
    val_cer, _, _ = evaluate(model, val_loader)
    scheduler.step(val_cer)
    history["train_loss"].append(train_loss); history["val_cer"].append(val_cer)
    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | val_CER {val_cer:.4f}")

    if val_cer < best_cer:
        best_cer = val_cer
        torch.save({"model": model.state_dict(), "chars": CFG.chars,
                    "img_height": CFG.img_height, "img_width": CFG.img_width,
                    "rnn_hidden": CFG.rnn_hidden}, ckpt_path)

print(f"Best validation CER: {best_cer:.4f}  ->  saved to {ckpt_path}")

## 8. Evaluation

We plot the learning curves and inspect a few validation predictions side-by-side with the truth. The best checkpoint is reloaded so the numbers below reflect the model we will actually submit.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history["train_loss"], marker="o"); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(history["val_cer"], marker="o", color="crimson"); ax[1].set_title("Validation CER"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

model.load_state_dict(torch.load(ckpt_path, map_location=CFG.device)["model"])
val_cer, preds, gts = evaluate(model, val_loader)
exact = np.mean([p == t for p, t in zip(preds, gts)])
print(f"Validation CER: {val_cer:.4f} | exact-match accuracy: {exact:.3f}")
for p, t in list(zip(preds, gts))[:12]:
    print(f"  pred={p:<8} truth={t:<8} {'OK' if p == t else 'X'}")

## 9. Predict on the test set & build the submission

We run the best model over the 5,000 test images and write `image,prediction` rows.

In [ ]:
def natural_key(name):
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else name

test_files = sorted([p.name for p in CFG.test_dir.glob("*.png")], key=natural_key)
test_ds = TextImageDataset(CFG.test_dir, filenames=test_files, transform=build_transforms(False))
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                         num_workers=CFG.num_workers, collate_fn=ctc_collate)

model.eval()
rows = []
with torch.no_grad():
    for imgs, _, _, names in tqdm(test_loader, desc="Predicting"):
        rows += list(zip(names, greedy_decode(model(imgs.to(CFG.device)).cpu())))

submission = pd.DataFrame(rows, columns=["image", "prediction"])
sub_path = CFG.out_dir / "submission.csv"
submission.to_csv(sub_path, index=False)
print("Saved:", sub_path, "| rows:", len(submission))
submission.head()

## 10. Conclusion & how to improve

**What we built.** A CRNN (CNN → BiLSTM → linear) trained with CTC loss that reads distorted 6-character codes end-to-end, evaluated with CER, and emits a ready-to-submit CSV.

**Ways model can be tweaked to improve CER:**

1. **Train longer / more data.** Increase `CFG.epochs` (e.g. 40–60) and keep the best-CER checkpoint.
2. **Stronger augmentation** to fight the occlusion blob and noise: add `RandomErasing`, blur, and noise. This usually helps generalisation the most here.
3. **Bigger/wider model.** Raise `CFG.rnn_hidden` (256 → 384/512) or `CFG.img_width` (160 → 200) for more time steps.
4. **Learning-rate schedule.** Try a `OneCycleLR` or cosine schedule; tune `CFG.lr` in `[3e-4, 2e-3]`.
5. **Decoding.** Swap greedy decoding for **beam search**; since every label is length 6, you can also keep only the 6 most confident non-blank positions.
6. **Ensembling / TTA.** Average predictions from a few seeds or light test-time augmentations.